# Assignment 2: Spam Email Detection Using NLP

In this assignment I make a small dataset of 10 emails (5 spam and 5 ham). Then I clean the text, turn it into numbers using Bag of Words, and make vector embeddings. The result of every step is shown below the code.

## Step 0: Install and Import Libraries

In [ ]:
# Run this cell once if the libraries are not installed
!pip install nltk gensim scikit-learn pandas

: 

In [ ]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from gensim.models import Word2Vec

# download the data that NLTK needs
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

# show full text and all columns when printing
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

**What is happening here**

- pandas is used to keep the emails in a table (DataFrame).
- numpy is used for the average of vectors in the last step.
- nltk gives us tokenizer, stop words, stemmer and lemmatizer.
- CountVectorizer from scikit-learn makes the Bag of Words.
- Word2Vec from gensim makes the word embeddings.
- `nltk.download` gets the word lists that NLTK needs. It is only downloaded the first time.
- The `pd.set_option` lines only change how the table is printed, so the full email text is visible.

## Step 1: Create the Dataset and Load it into a DataFrame

I made the 10 emails from the sentences of the sample paragraph. Five are spam (prize, lottery, free offers) and five are ham (assignment, class, lecture notes, meeting, feedback). The fifth spam email joins words from the spam sentences of the paragraph, so that I get 5 spam emails.

In [ ]:
emailList = [
    "Congratulations! You have won a free prize. Click here to claim your reward.",
    "Get a free mobile phone now.",
    "Claim your free gift card before the offer expires.",
    "You have won a cash lottery.",
    "Click here to claim your free cash reward.",
    "Your assignment deadline is Friday. Please submit it on time.",
    "Your class will start tomorrow at 9 AM.",
    "Please find the lecture notes attached.",
    "Thank you for attending the meeting.",
    "Please share your feedback."
]

labelList = ["spam", "spam", "spam", "spam", "spam",
             "ham", "ham", "ham", "ham", "ham"]

df = pd.DataFrame({"message": emailList, "label": labelList})

print(df)

**What is happening here**

- `emailList` holds the text of the 10 emails and `labelList` holds the label of each email in the same order.
- `pd.DataFrame` joins both lists into a table with two columns: `message` and `label`.
- Each row is one email. Row 0 to 4 are spam and row 5 to 9 are ham.

## Step 2: Exploratory Data Analysis (EDA)

In [ ]:
print("First 5 rows (head):")
print(df.head())

print()
print("Shape (rows, columns):")
print(df.shape)

print()
print("Info:")
df.info()

print()
print("Number of spam and ham emails:")
print(df["label"].value_counts())

**What is happening here**

- `head()` shows the first 5 rows, so we can quickly see how the data looks.
- `shape` gives (rows, columns). Here it is (10, 2), which means 10 emails and 2 columns.
- `info()` shows the column names, how many values are not empty, and the data type. Both columns are text (object) and there are no missing values.
- `value_counts()` counts each label. We get 5 spam and 5 ham, so the data is balanced.

## Step 3: Text Preprocessing

Raw text is messy, so we clean it in small steps. After each step the result is printed.

In [ ]:
# things we need for the next steps
stopWordSet = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

### 3.1 Lowercasing

In [ ]:
df["lowerText"] = df["message"].str.lower()

print(df[["lowerText"]])

**What is happening here**

`str.lower()` changes every letter to small letters. This is done so that "Free" and "free" are counted as the same word.

### 3.2 Tokenization

In [ ]:
def getTokens(text):
    tokenList = word_tokenize(text)
    cleanList = []
    for word in tokenList:
        # keep only words, this removes marks like ! and . and numbers like 9
        if word.isalpha():
            cleanList.append(word)
    return cleanList

df["tokens"] = df["lowerText"].apply(getTokens)

print(df[["tokens"]])

**What is happening here**

- `word_tokenize` breaks a sentence into separate words (tokens).
- The loop keeps a token only if `isalpha()` is True, so punctuation and numbers are dropped.
- `apply` runs the function on every row of the column.

### 3.3 Stop Word Removal

In [ ]:
def removeStopWords(tokenList):
    resultList = []
    for word in tokenList:
        if word not in stopWordSet:
            resultList.append(word)
    return resultList

df["noStopWords"] = df["tokens"].apply(removeStopWords)

print(df[["noStopWords"]])

**What is happening here**

Stop words are common words like "you", "the", "is", "to". They are in almost every email, so they do not help to tell spam from ham. The function checks each word and keeps it only if it is not in the NLTK stop word list.

### 3.4 Stemming

In [ ]:
def getStems(tokenList):
    resultList = []
    for word in tokenList:
        resultList.append(stemmer.stem(word))
    return resultList

df["stemmed"] = df["noStopWords"].apply(getStems)

print(df[["noStopWords", "stemmed"]])

**What is happening here**

Stemming cuts the end of a word to get its root. The Porter stemmer works by rules, so the root is not always a real word. For example, "congratulations" becomes "congratul" and "expires" becomes "expir".

### 3.5 Lemmatization

In [ ]:
def getLemmas(tokenList):
    resultList = []
    for word in tokenList:
        # pos="v" means we treat the word as a verb, so "won" becomes "win"
        resultList.append(lemmatizer.lemmatize(word, pos="v"))
    return resultList

df["lemmas"] = df["noStopWords"].apply(getLemmas)

print(df[["noStopWords", "lemmas"]])

**What is happening here**

Lemmatization also finds the base form of a word, but it uses a dictionary (WordNet), so the result is always a real word. For example, "won" becomes "win" and "expires" becomes "expire". I used `pos="v"` so the words are treated as verbs. That is why verbs like "won" are fixed.

Stemming is faster, but lemmatization gives cleaner words. For the next steps I use the lemmatized words.

In [ ]:
print("Final result of preprocessing:")
print(df[["message", "lemmas"]])

## Step 4: Feature Extraction Using Bag of Words (BoW)

In [ ]:
def joinWords(tokenList):
    return " ".join(tokenList)

# CountVectorizer needs text, so join the lemmas back into one string
df["cleanText"] = df["lemmas"].apply(joinWords)

vectorizer = CountVectorizer()
bowMatrix = vectorizer.fit_transform(df["cleanText"])

wordList = vectorizer.get_feature_names_out()
bowDf = pd.DataFrame(bowMatrix.toarray(), columns=wordList)

print("Clean text of each email:")
print(df[["cleanText"]])

print()
print("Vocabulary (all unique words):")
print(wordList)

print()
print("Number of unique words:", len(wordList))
print("Shape of BoW matrix:", bowMatrix.shape)

print()
print("Bag of Words table (row = email, column = word):")
print(bowDf)

**What is happening here**

- The lemmas of each email are joined back into one string, because `CountVectorizer` takes text as input.
- `fit_transform` first reads all emails and makes a list of unique words (the vocabulary). Then it counts how many times each word appears in each email.
- Each row of the BoW table is one email and each column is one word. The number is the count of that word in that email. Most numbers are 0, because an email uses only a few words from the full vocabulary.
- The shape is (10, number of unique words).
- Words like "free", "claim", "win" and "reward" appear in the spam rows, while words like "please", "class" and "meet" appear in the ham rows. This is what a spam model can learn from.

## Step 5: Vector Embeddings

In [ ]:
sentenceList = df["lemmas"].tolist()

# train a small Word2Vec model on our 10 emails
w2vModel = Word2Vec(sentences=sentenceList, vector_size=50, window=3,
                    min_count=1, workers=1, seed=42, epochs=100)

print("Number of words in the Word2Vec model:", len(w2vModel.wv))
print()
print("Vector of the word 'free' (first 10 values):")
print(w2vModel.wv["free"][:10])

**What is happening here**

- Word2Vec turns each word into a list of numbers (a vector). Words that are used in similar places get similar vectors.
- `sentences` is our list of lemmatized emails, where each email is a list of words.
- `vector_size=50` means each word gets 50 numbers.
- `window=3` means the model looks at 3 words on each side of a word.
- `min_count=1` keeps every word, even if it appears only once. We need this because our data is very small.
- `epochs=100` means the model goes through the data 100 times, so it learns better from the small data.
- `seed=42` and `workers=1` are used so that we get the same result every time we run the code.

In [ ]:
def getEmailVector(tokenList):
    vectorList = []
    for word in tokenList:
        vectorList.append(w2vModel.wv[word])
    # average of all word vectors gives one vector for the email
    return np.mean(vectorList, axis=0)

emailVectorList = []
for tokenList in sentenceList:
    emailVectorList.append(getEmailVector(tokenList))

embedDf = pd.DataFrame(emailVectorList)

print("Shape of email embeddings:", embedDf.shape)
print()
print("Embedding of each email (first 6 values of 50):")
print(embedDf.iloc[:, :6])

**What is happening here**

- Word2Vec gives one vector for each word, but we want one vector for each email.
- `getEmailVector` takes the vector of every word in the email and finds the average using `np.mean`. This average is the embedding of the email.
- The loop does this for all 10 emails and the results are kept in a DataFrame.
- The shape is (10, 50): 10 emails and 50 numbers for each email.
- Only the first 6 of the 50 values are printed so the output stays short.
- Note: the model is trained on only 10 short emails, so the vectors are only for practice. In real work, a model trained on a large amount of text is used.

## Summary

I made a dataset of 10 emails (5 spam, 5 ham) and loaded it into a Pandas DataFrame. EDA with `head()`, `shape` and `info()` showed 10 rows, 2 columns and no missing values. The text was cleaned by lowercasing, tokenization, stop word removal, stemming and lemmatization. Bag of Words turned the cleaned emails into word counts, and Word2Vec gave a 50 number vector for each email.